<div style="background-color:#1F3864; padding:25px; border-radius:8px;">
<h1 style="color:white; text-align:center; margin:0;">🌍 Global Trade Disruption & Commodity Price Prediction</h1>
<p style="color:#D9E2F3; text-align:center; margin-top:10px; font-size:15px;">
Predicting commodity price movements from real-world supply chain disruption signals -
built entirely on live API data (FRED, GDELT, UN Comtrade, World Bank)
</p>
</div>

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import time

## Notebook Overview

This notebook builds a dataset and model to predict commodity price movements —
starting with WTI crude oil — using real-world supply chain disruption signals
sourced directly from public APIs, rather than a synthetic dataset.

**Data sources:**
- **FRED API** — commodity prices (oil WTI, oil Brent, natural gas, wheat, corn, gold, aluminum, iron ore)
- **GDELT** — geopolitical event intensity and global news coverage volume
- **UN Comtrade** — bilateral trade volume between countries
- **World Bank** — country-level logistics performance and economic context

**Target variable:**
Percentage change in WTI crude oil price over a forward window (e.g. 30 days),
predicted from current disruption signals. Predicting the *change* rather than
the raw price level avoids the model simply learning long-term inflation trends,
and instead focuses on how disruption events actually move prices — the same
approach used in the crash prediction project's 63-day-forward target.

**Major real-world events this data is expected to capture:**
COVID-19 Supply Chain Shock (2020), Russia-Ukraine Conflict (2022),
Red Sea Shipping Crisis (2024), Strait of Hormuz Disruption (2026).

<div style="background-color:#EDCC80; padding:15px; border-radius:8px;">
<h2 style="color:#1F3864; text-align:center; margin:0;">1. Create the Data</h2>
</div>

<div style="background-color:#EDEAE5; padding:10px 15px; border-radius:6px;">
<h3 style="color:#1F3864; text-align:center; margin:0;">1.1 Commodities</h3>
</div>

In [2]:
api_key = "a3b1c076b07a348c2f9a24b0799e53e2"

In [3]:
start_date = "2000-01-01"

In [4]:
commodities = {
    "oil_wti": "DCOILWTICO",
    "oil_brent": "DCOILBRENTEU",
    "natural_gas": "DHHNGSP",
    "wheat": "PWHEAMTUSDM",
    "corn": "PMAIZMTUSDM",
    "gold": "IQ12260",
    "aluminum": "PALUMUSDM",
    "iron_ore": "PIORECRUSDM"}

In [5]:
def get_commodity_prices(commodities:dict,start_date:str)-> None :
    for name,series_id in commodities.items():
        url = f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&observation_start={start_date}&api_key={api_key}&file_type=json"
        response = requests.get(url)

        try:
            data = response.json()
        except ValueError:
            print(f"FAILED: {name} ({series_id}) — empty or invalid response, status {response.status_code}")
            continue

        if "observations" not in data:
            print(f"FAILED: {name} ({series_id}) — {data.get('error_message', 'unknown error')}")
            continue

        df = pd.DataFrame(data["observations"])
        df = df[["date", "value"]]
        df.to_csv(f"data/{name}.csv", index=False)
        print(f"Saved {len(df)} rows to data/{name}.csv")

        time.sleep(0.5)

In [6]:
get_commodity_prices(commodities,start_date)

Saved 6931 rows to data/oil_wti.csv
Saved 6931 rows to data/oil_brent.csv
Saved 6931 rows to data/natural_gas.csv
Saved 318 rows to data/wheat.csv
Saved 318 rows to data/corn.csv
Saved 318 rows to data/gold.csv
Saved 318 rows to data/aluminum.csv
Saved 318 rows to data/iron_ore.csv


<div style = background-color:#EDEAE5;padding:10px 15px;border-radius:6px;">
<h3 style = "color:#1F3864;text-align:center;margin:0">1.2 GDELT Events </h3>
</div>

In [7]:
def get_gdelt_events(queries: list) -> None:
    headers = {"User-Agent": "Mozilla/5.0"}

    for query in queries:
        url = "https://api.gdeltproject.org/api/v2/doc/doc"
        params = {
            "query": f'"{query}"',
            "mode": "timelinevol",
            "format": "json"
        }
        response = requests.get(url, params=params, headers=headers)

        if response.status_code == 429:
            print(f"Rate limited on {query}, waiting...")
            time.sleep(5)
            response = requests.get(url, params=params, headers=headers)

        try:
            data = response.json()
        except ValueError:
            print(f"FAILED: {query} — status {response.status_code}, invalid response")
            continue

        if "timeline" not in data:
            print(f"FAILED: {query} — keys were {list(data.keys())}")
            continue

        records = data["timeline"][0]["data"]
        df = pd.DataFrame(records)
        filename = query.lower().replace(" ", "_")
        df.to_csv(f"data/gdelt_{filename}.csv", index=False)
        print(f"Saved data/gdelt_{filename}.csv")

        time.sleep(2)

events = ["Strait of Hormuz", "Red Sea shipping", "Russia Ukraine conflict", "COVID supply chain"]

In [8]:
get_gdelt_events(events)

Rate limited on Strait of Hormuz, waiting...
FAILED: Strait of Hormuz — status 429, invalid response
Rate limited on Red Sea shipping, waiting...
FAILED: Red Sea shipping — status 429, invalid response
Rate limited on Russia Ukraine conflict, waiting...
FAILED: Russia Ukraine conflict — status 429, invalid response
Rate limited on COVID supply chain, waiting...
FAILED: COVID supply chain — status 429, invalid response


In [9]:
time.sleep(30)
get_gdelt_events(["Red Sea shipping"])

Rate limited on Red Sea shipping, waiting...
FAILED: Red Sea shipping — status 429, invalid response


<div style = background-color:#EDEAE5;padding:10px 15px;border-radius:6px;">
<h3 style = "color:#1F3864;text-align:center;margin:0">1.3 UN-Comtrade </h3>
</div>

In [10]:
primary_key = "fb672033c45d4d71a329a740d788c758"

In [11]:
reporters = {
    "egypt": "818",
    "iran": "364",
    "china": "156",
    "usa": "842"}

In [12]:
def get_comtrade_data(reporters: dict, start_year: int, end_date: int) -> None:
    url = "https://comtradeapi.un.org/data/v1/get/C/A/HS"
    headers = {"Ocp-Apim-Subscription-Key": primary_key}

    all_years = list(range(start_year, end_date + 1))
    year_chunks = [all_years[i:i+12] for i in range(0, len(all_years), 12)]

    for name, reporter_code in reporters.items():
        all_records = []

        for chunk in year_chunks:
            years_str = ",".join(str(y) for y in chunk)
            params = {
                "reporterCode": reporter_code,
                "partnerCode": "0",
                "period": years_str,
                "cmdCode": "TOTAL",
                "flowCode": "M"
            }

            response = requests.get(url, headers=headers, params=params)

            if response.status_code == 429:
                print(f"Rate limited on {name}, waiting...")
                time.sleep(3)
                response = requests.get(url, headers=headers, params=params)

            try:
                data = response.json()
            except ValueError:
                print(f"FAILED chunk {years_str} for {name} — invalid response")
                continue

            if "data" not in data:
                print(f"FAILED chunk {years_str} for {name} — {data.get('error')}")
                continue

            all_records.extend(data["data"])
            time.sleep(1.5)

        df = pd.DataFrame(all_records)
        df.to_csv(f"data/comtrade_{name}.csv", index=False)
        print(f"Saved {len(df)} rows to data/comtrade_{name}.csv")

In [13]:
get_comtrade_data(reporters, start_year=2000, end_date = 2027)

Saved 26 rows to data/comtrade_egypt.csv
Saved 19 rows to data/comtrade_iran.csv
Saved 658 rows to data/comtrade_china.csv
Saved 26 rows to data/comtrade_usa.csv


<div style = background-color:#EDEAE5;padding:10px 15px;border-radius:6px;">
<h3 style = "color:#1F3864;text-align:center;margin:0">1.4 World Bank </h3>
</div>

In [14]:
indicators = {
    "logistics_performance": "LP.LPI.OVRL.XQ",
    "gdp": "NY.GDP.MKTP.CD",
    "trade_pct_gdp": "NE.TRD.GNFS.ZS"}

In [15]:
def get_worldbank_data(reporters: dict, indicators: dict) -> None:
    for country_name, country_code in reporters.items():
        all_records = []

        for indicator_name, indicator_code in indicators.items():
            url = "https://api.worldbank.org/v2/country/EGY/indicator/LP.LPI.OVRL.XQ?format=json"
            response = requests.get(url)

            try:
                data = response.json()
            except ValueError:
                print(f"FAILED: {country_name} / {indicator_name} — invalid response")
                continue

            if len(data) < 2 or data[1] is None:
                print(f"FAILED: {country_name} / {indicator_name} — no data returned")
                continue

            records = data[1]
            for r in records:
                r["indicator_name"] = indicator_name
            all_records.extend(records)

            time.sleep(0.5)

        df = pd.DataFrame(all_records)
        df.to_csv(f"data/worldbank_{country_name}.csv", index=False)
        print(f"Saved {len(df)} rows to data/worldbank_{country_name}.csv")

In [16]:
get_worldbank_data(reporters, indicators)

Saved 150 rows to data/worldbank_egypt.csv
Saved 150 rows to data/worldbank_iran.csv
Saved 150 rows to data/worldbank_china.csv
Saved 150 rows to data/worldbank_usa.csv


<div style="background-color:#EDCC80; padding:15px; border-radius:8px;">
<h2 style="color:#1F3864; text-align:center; margin:0;">2. Load the data </h2>
</div>

<div style = background-color:#EDEAE5;padding:10px 15px;border-radius:6px;">
<h3 style = "color:#1F3864;text-align:center;margin:0">2.1 Commodities </h3>
</div>

In [17]:
oil_wti = pd.read_csv("data/oil_wti.csv")
oil_brent = pd.read_csv("data/oil_brent.csv")
natural_gas = pd.read_csv("data/natural_gas.csv")
wheat = pd.read_csv("data/wheat.csv")
corn = pd.read_csv("data/corn.csv")
gold = pd.read_csv("data/gold.csv")
aluminum = pd.read_csv("data/aluminum.csv")
iron_ore = pd.read_csv("data/iron_ore.csv")

<div style = background-color:#EDEAE5;padding:10px 15px;border-radius:6px;">
<h3 style = "color:#1F3864;text-align:center;margin:0">2.2 gdelt </h3>
</div>

In [18]:
gdelt_hormuz = pd.read_csv("data/gdelt_strait_of_hormuz.csv")
gdelt_red_sea = pd.read_csv("data/gdelt_red_sea_shipping.csv")
gdelt_russia_ukraine = pd.read_csv("data/gdelt_russia_ukraine_conflict.csv")
gdelt_covid = pd.read_csv("data/gdelt_covid_supply_chain.csv")

<div style = background-color:#EDEAE5;padding:10px 15px;border-radius:6px;">
<h3 style = "color:#1F3864;text-align:center;margin:0">2.3 UN-Comtrade </h3>
</div>

In [21]:
comtrade_egypt = pd.read_csv("data/comtrade_egypt.csv")
comtrade_iran = pd.read_csv("data/comtrade_iran.csv")
comtrade_china = pd.read_csv("data/comtrade_china.csv")
comtrade_usa = pd.read_csv("data/comtrade_usa.csv")

<div style = background-color:#EDEAE5;padding:10px 15px;border-radius:6px;">
<h3 style = "color:#1F3864;text-align:center;margin:0">2.4 World Bank </h3>
</div>

In [23]:
worldbank_egypt = pd.read_csv("data/worldbank_egypt.csv")
worldbank_iran = pd.read_csv("data/worldbank_iran.csv")
worldbank_china = pd.read_csv("data/worldbank_china.csv")
worldbank_usa = pd.read_csv("data/worldbank_usa.csv")

<div style="background-color:#EDCC80; padding:15px; border-radius:8px;">
<h2 style="color:#1F3864; text-align:center; margin:0;">3. Merge the data </h2>
</div>